# Semantic Classifier

Architecture ideas:
1. Checkpoints
- all-MiniLM-L6-v2 (Transformer encoder)
2. Machine Learning
- Linear Regression 
- CNN 

## all-MiniLM-L6-v2 Details
- Base Model: Fine-tuned from the (nreimers/MiniLM-L6-H384-uncased checkpoint)[https://arxiv.org/pdf/2002.10957].
- Layers: 6 transformer layers.
- Hidden Dimension: 384 dimensions.
- Attention Heads: 12 attention heads.
- Parameter Count: Approximately 22.7 million parameters.
- Max Sequence Length: 128 tokens (inputs are typically truncated past 256 word pieces depending on configuration).

Architecture:
1. Tokenizer: uncased WordPiece, 30 522 vocab, 256 max char [alg.](https://huggingface.co/learn/llm-course/en/chapter6/6)

2. Embeddings:
- Word embeddings (30 522 * 384)
- Position embeddings (512 * 384)
- Token type embeddings (2 * 384)
- LayerNorm (eps 1e-12)
- Dropout

3. Encoder
- Self-attention
    - Q, K, V linear layers (384 to 384)
    - 12 heads by 32 dims each
    - Dropout
    - Residual connection
    - LayerNorm
- Intermediate (feed forward, up)
    - Dense (384 to 1536)
    - GELU
- Output (feed forward, down)
    - Dense (1536 to 384)
    - Dropout
    - Residual connection
    - LayerNorm

4. Attention Masked Mean Pooling over Token Outputs

5. L2 Normalization on Pooled Vector

6. Classifier Head (384 to x categories)

## Tokenizer: WordPiece
Source: Hugging Face

In [1]:
from transformers import AutoTokenizer

Import Vocabulary

In [17]:
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

vocab = tokenizer.get_vocab()

print(len(vocab))
print(vocab)

30522
{'##vira': 24093, 'causing': 4786, 'silence': 4223, '##chin': 17231, '##tist': 16774, 'fiery': 15443, 'highlights': 11637, '##encies': 15266, 'tumor': 13656, 'bihar': 16178, '##nsk': 25564, 'inherit': 22490, 'endorsed': 11763, '##mut': 28120, '##tford': 26341, '安': 1820, 'caucasus': 16512, 'waved': 7147, 'republican': 3951, '[unused545]': 550, '##ronia': 28140, 'spared': 16891, 'fiance': 19154, 'grid': 8370, 'terribly': 16668, 'drops': 9010, 'retailers': 16629, '##ations': 10708, 'stresses': 23253, 'eleanor': 10508, 'greene': 11006, 'parent': 6687, 'stroll': 27244, 'kellan': 19697, 'grin': 5861, '書': 1871, 'pray': 11839, 'acquaintance': 18363, '##rity': 15780, '##gn': 16206, '##illa': 9386, '[unused20]': 21, '##vah': 21927, 'roadway': 18799, 'controls': 7711, 'assassinate': 25683, '[unused689]': 694, 'asean': 27974, 'heart': 2540, 'strode': 11885, 'firstly': 15847, 'participating': 8019, '##cel': 29109, 'horseback': 19926, 'italics': 19408, 'foraging': 26912, 'cathedral': 5040, '

Save Vocabulary

In [18]:
tokenizer.save_pretrained("./minilm_tokenizer")

('./minilm_tokenizer\\tokenizer_config.json',
 './minilm_tokenizer\\tokenizer.json')

Pre-tokenizaton:
1. Lower
2. Remove accents
3. Split punctuaton
4. Split whitespace

In [24]:
import re

bio = "I'm a software engineer passionate about machine learning, optimization, and discovering new ways to solve problems. Across my industry and hospitality experience, I learned to adapt and coordinate with cross-functional teams in fast-paced environments. Currently seeking software and AI engineering graduate roles to make a meaningful impact."

bio = bio.lower()
bio = re.split(r"([^\s\w])", bio)

bio = [word for words in bio for word in words.split()]

print(bio)

['i', "'", 'm', 'a', 'software', 'engineer', 'passionate', 'about', 'machine', 'learning', ',', 'optimization', ',', 'and', 'discovering', 'new', 'ways', 'to', 'solve', 'problems', '.', 'across', 'my', 'industry', 'and', 'hospitality', 'experience', ',', 'i', 'learned', 'to', 'adapt', 'and', 'coordinate', 'with', 'cross', '-', 'functional', 'teams', 'in', 'fast', '-', 'paced', 'environments', '.', 'currently', 'seeking', 'software', 'and', 'ai', 'engineering', 'graduate', 'roles', 'to', 'make', 'a', 'meaningful', 'impact', '.']


Tokenize

In [29]:
def tokenize(word):
    start = 0
    tokens = []
    while start < len(word):
        end = len(word)
        found = False

        while end > start:
            substring = word[start:end]

            if start > 0:
                substring = "##" + substring

            if substring in vocab:
                tokens.append(substring)
                start = end
                found = True
                break

            end -= 1

        if not found:
            return ["[UNK]"]

    return tokens

Token ID

In [79]:
max_tokens = 256

bio_tokens = ["[CLS]"]

for word in bio:
    bio_tokens += tokenize(word)

if len(bio_tokens) > max_tokens - 1:  # Reserve space for [SEP]
    bio_tokens = bio_tokens[:max_tokens - 1]

bio_tokens += ["[SEP]"]

token_ids = [vocab.get(token, vocab["[UNK]"]) for token in bio_tokens]

print(token_ids)

[101, 1045, 1005, 1049, 1037, 4007, 3992, 13459, 2055, 3698, 4083, 1010, 20600, 1010, 1998, 13648, 2047, 3971, 2000, 9611, 3471, 1012, 2408, 2026, 3068, 1998, 15961, 3325, 1010, 1045, 4342, 2000, 15581, 1998, 13530, 2007, 2892, 1011, 8360, 2780, 1999, 3435, 1011, 13823, 10058, 1012, 2747, 6224, 4007, 1998, 9932, 3330, 4619, 4395, 2000, 2191, 1037, 15902, 4254, 1012, 102]


## Embedding

In [80]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Get the underlying Transformer model
transformer = model[0].auto_model

# Word/token embedding weights
word_weights = transformer.embeddings.word_embeddings.weight

# Position embedding weights
position_weights = transformer.embeddings.position_embeddings.weight

# Token type embedding weights
token_type_weights = transformer.embeddings.token_type_embeddings.weight

print(word_weights.shape)
print(position_weights.shape)
print(token_type_weights.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

torch.Size([30522, 384])
torch.Size([512, 384])
torch.Size([2, 384])


In [81]:
import torch

token_ids = torch.tensor(token_ids)

token_ids = token_ids.unsqueeze(0)

In [82]:
word_embeddings = word_weights[token_ids]
position_embeddings = position_weights[:len(token_ids)]
token_type_embeddings = token_type_weights[0].expand(len(token_ids), -1)

In [83]:
print(word_embeddings.shape)
print(position_embeddings.shape)
print(token_type_embeddings.shape)

torch.Size([1, 61, 384])
torch.Size([1, 384])
torch.Size([1, 384])


In [84]:
x = word_embeddings + position_embeddings + token_type_embeddings

In [85]:
embeddings = transformer.embeddings

LayerNorm

In [86]:
x = embeddings.LayerNorm(x)

Dropout

In [87]:
x = embeddings.dropout(x)  # no effect in eval mode

In [88]:
print("token_ids:", token_ids.shape)
print("x:", x.shape)
print("Q:", Q.shape)

token_ids: torch.Size([1, 61])
x: torch.Size([1, 61, 384])
Q: torch.Size([1, 1, 1, 61, 384])


## Encoder

In [91]:
import torch
from torch.nn.functional import softmax

encoder_layers = transformer.encoder.layer

for layer in encoder_layers:

    # ==========================================
    # SELF ATTENTION
    # ==========================================

    attention = layer.attention.self

    Q = attention.query(x)
    K = attention.key(x)
    V = attention.value(x)

    batch_size, seq_len, hidden_size = Q.shape

    num_heads = attention.num_attention_heads
    head_dim = attention.attention_head_size

    # [batch, seq, 384]
    #        ↓
    # [batch, 12, seq, 32]

    Q = Q.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
    K = K.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)

    # Attention scores
    scores = Q @ K.transpose(-2, -1)
    scores = scores / (head_dim ** 0.5)

    attention_probs = softmax(scores, dim=-1)

    # Weighted sum of V
    context = attention_probs @ V

    # [batch, 12, seq, 32]
    #        ↓
    # [batch, seq, 384]

    context = context.transpose(1, 2).contiguous()
    context = context.view(batch_size, seq_len, hidden_size)

    # Attention output projection
    context = layer.attention.output.dense(context)

    # Dropout
    context = layer.attention.output.dropout(context)

    # Residual + LayerNorm
    x = layer.attention.output.LayerNorm(x + context)


    # ==========================================
    # FEED FORWARD
    # ==========================================

    # 384 -> 1536 + GELU
    ff = layer.intermediate(x)

    # 1536 -> 384
    ff = layer.output.dense(ff)

    # Dropout
    ff = layer.output.dropout(ff)

    # Residual + LayerNorm
    x = layer.output.LayerNorm(x + ff)

In [92]:
x.shape

torch.Size([1, 61, 384])

## Attention-masked mean pooling

In [94]:
attention_mask = torch.ones_like(token_ids)
print("attention_mask:", attention_mask.shape)

attention_mask: torch.Size([1, 61])


In [96]:
mask = attention_mask.to(x.device).unsqueeze(-1).float()

In [97]:
masked_x = x * mask

sum_embeddings = masked_x.sum(dim=1)
sum_mask = mask.sum(dim=1)

pooled = sum_embeddings / sum_mask.clamp(min=1e-9)

sentence_embedding = torch.nn.functional.normalize(
    pooled,
    p=2,
    dim=1
)

print(sentence_embedding.shape)
print(torch.norm(sentence_embedding, p=2, dim=1))

torch.Size([1, 384])
tensor([1.], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)


## L2 Norm

In [98]:
# L2 normalization
sentence_embedding = torch.nn.functional.normalize(
    pooled,
    p=2,
    dim=1
)

## Output

In [99]:
print("Final embedding:", sentence_embedding.shape)
print("L2 norm:", torch.norm(sentence_embedding, p=2, dim=1))

Final embedding: torch.Size([1, 384])
L2 norm: tensor([1.], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
